# Reproducing Parisi & Rutkowski's Ac-225 headline from muon-catalyzed fusion

**Paper:** J. F. Parisi & A. Rutkowski, *Isotope Production in Muon-Catalyzed-Fusion Systems*,
[arXiv:2511.20951](https://arxiv.org/abs/2511.20951) (Marathon Fusion, 2025).

This notebook mirrors the CI-tested script `scripts/parisi_ac225.py`. It reproduces the paper's headline
&mdash; a &mu;CF neutron source with a **10 g `226Ra` feedstock** and a steady-state **`1e12` muons/s**
rate (**~0.5 kW** of D-T fusion power) makes **~20 mg/yr of `225Ac`**, "comparable to 400 times global
supply in 2024" &mdash; as a genuine **forward** computation from the paper's own published factors. No
factor is back-solved to hit 20 mg/yr (invariant I2: a discrepancy would be a documented finding,
not a fudge).

Transmutation pathway (paper eq. (28)):

$$^{226}\mathrm{Ra}(n,2n)\,^{225}\mathrm{Ra}\ \xrightarrow[\ 15\ \mathrm{d}\ ]{\ \beta^-\ }\ ^{225}\mathrm{Ac}$$

**Positioning (invariant I8/I9):** this is a reproduction of an *external* group's result, not an OpenMuCF
claim. The "viable well before energy breakeven" language quoted at the end is Parisi & Rutkowski's.

## 1. The factor chain (loaded from the CI-tested script)

Every factor carries a locator into arXiv:2511.20951v2 (or a standard constant source). The tests in `tests/test_parisi.py` assert that each row has a non-empty citation (invariant I3).

In [ ]:
import pathlib
import sys

# Locate the repo root (works whether the notebook is launched from repo root or notebooks/).
_here = pathlib.Path.cwd()
for _base in (_here, *_here.parents):
    if (_base / "scripts" / "parisi_ac225.py").exists():
        sys.path.insert(0, str(_base / "scripts"))
        break

import parisi_ac225 as P  # noqa: E402  (import follows the sys.path setup above)

for f in P.FACTORS:
    print(f"{f.symbol:<11} = {f.value:.6g} {f.unit:<12}  ({f.name})")
    print(f"    cite: {f.citation}")

## 2. The forward computation

`Ndot_n = R_mu * N_fus_mu` (eqs. 3, A9) &rarr; `Ndot_225 = eta_pro * Ndot_n` (eq. 39) &rarr; `Mdot = Ndot_225 * (A_Ac/N_A) * s_year` (eq. 14). The fusion power `P_fus = Ndot_n * E_fus` (eq. 10) is a published cross-check.

In [ ]:
r = P.reproduce()
print(f"Ndot_n   = R_mu * N_fus_mu            = {r['Ndot_n_per_s']:.4e} neutrons/s   [eq. (3),(A9)]")
print(f"P_fus    = Ndot_n * E_fus             = {r['P_fus_W']:.1f} W              "
      f"[eq. (10); Table I: {P.TABLE_I_PFUS_W:.0f} W]")
print(f"Ndot_225 = eta_pro * Ndot_n           = {r['Ndot_225Ac_per_s']:.4e} atoms/s     [eq. (39)]")
print(f"m_pro    = A_Ac / N_A                 = {r['m_pro_g']:.4e} g/atom      [eq. (14)]")
print(f"Mdot     = Ndot_225 * m_pro * s_year  = {r['Ac225_mg_per_year']:.3f} mg/yr "
      f"(= {r['Ac225_ug_per_year']:.0f} ug/yr)")

## 3. Reproduction vs the paper (+ published cross-checks)

In [ ]:
print(f"headline (abstract): {P.HEADLINE_MG_PER_YEAR:.0f} mg/yr   -> forward = "
      f"{r['Ac225_mg_per_year']:.2f} mg/yr  ({r['dev_vs_headline_pct']:+.2f} %)")
print(f"Table I (OpenMC):    {P.TABLE_I_UG_PER_YEAR:.0f} ug/yr -> forward = "
      f"{r['Ac225_ug_per_year']:.0f} ug/yr ({r['dev_vs_table1_pct']:+.2f} %)")
print(f"vs 2024 global supply ({P.GLOBAL_2024_UG_PER_YEAR:.0f} ug/yr, ref. [33]): "
      f"{r['global_supply_multiple']:.0f}x   (abstract: ~{P.GLOBAL_2024_MULTIPLE:.0f}x)")
_within = abs(r['dev_vs_headline_pct']) <= P.RECON_TOL_FRAC * 100.0
print(f"=> within +/-{P.RECON_TOL_FRAC*100:.0f}% of headline: {'YES' if _within else 'NO'}")

## 4. Muon source &rarr; `MUON_COST.md` tier cross-reference

The `1e12` muons/s is a **rate** assumption; `MUON_COST.md` tiers classify muon **cost** (GeV/muon), so the *cost* maps to Tier 1 (design studies) while the *rate* itself is hypothetical &mdash; above every real facility.

In [ ]:
import textwrap

print(f"1e12 muons/s source, cost tier = {P.MUON_COST_TIER}")
print(textwrap.fill(P.MUON_COST_TIER_NOTE, width=96))

## 5. Framing (invariant I9) &mdash; Parisi & Rutkowski's claim, quoted as theirs

Below energy breakeven, &mu;CF's utility is **neutron-source / medical-isotope economics**, not
electricity. Selling the 14.1 MeV D-T neutrons as `225Ac` (a targeted-alpha-therapy isotope in chronic
short supply) relaxes the per-muon requirements by orders of magnitude: Parisi & Rutkowski report the
breakeven fusions-per-muon falling from `N_fus,mu >~ 415` (electricity-only) to `N_fus,mu >~ 5e-7`
(`225Ac` transmutation) [Sec III, Fig. 2].

On that basis **they** argue that &mu;CF *"systems employing transmutation could be viable well before
energy breakeven is possible"* and that the finding motivates muon-source development *"far before net
energy generation is possible"* [arXiv:2511.20951v2, abstract + Sec V].

OpenMuCF reproduces their arithmetic; **the viability judgement is theirs, not ours.**